In [ ]:
from eregion.tasks import ImageCreator, AssembleFocalPlane
from eregion.tasks.custom import guess_image_type_from_filename_DEIMOS, load_image_fits_DEIMOS
from eregion.tasks.calibration import MasterBias, CalibrationResult
from eregion.tasks.preprocessing import BiasSubtraction, ScanSubtraction, SigmaClipMasking

import numpy as np
import os
import glob2
import importlib

# Example: DEIMOS science focal plane characterization: PTC

In [ ]:
basepath = '/Users/yashvi/Desktop/Detector Characterization Tools/DTU_dettest/DTU_fullfp_bringup/PTC/SCI'
runid = '20260718*'
rawpath = glob2.glob(os.path.join(basepath, runid))[0]
outpath = rawpath.replace('/DTU_dettest/','/DTU_detreduce/')
if not os.path.exists(outpath):
    os.makedirs(outpath)

### Calproc

In [ ]:
# Load bias images
creator = ImageCreator(detector_config = '../src/eregion/configs/detectors/deimos_sci.yaml')
res = creator.run(input_source = os.path.join(rawpath,'*bias*.fits'),
                  identifier_func = guess_image_type_from_filename_DEIMOS,
                  fileloader_func = load_image_fits_DEIMOS,
                  data_on_demand = True)

# subtract overscan
oscan_sub = ScanSubtraction(which_scan='serial_overscan', method='median_by_axis', trim_start=6)
res = oscan_sub.run(images=res.data('type == "bias"'))

# make master bias
master_bias_task = MasterBias(method='median')
mb_res = master_bias_task.run(images=res.data('type == "bias"'))


In [ ]:
# cleanup memory
del res
# save master bias result obj
mb_res.save(outpath)

In [ ]:
# load master bias result obj
mb_res = CalibrationResult.load(outpath)
mb_res.master_bias

Make focal plane image. `AssembleFocalPlane` task can take ImageBundle and group them into sets of focal plane images based on the list of columns that uniquely identify an exposure.

In [ ]:
def plot_focal_plane(bundle, grouping_columns=['type', 'exptime', 'seqnum'], how_many=1, zscale=False, with_mask=False, **imshow_kwargs):
    fp_task = AssembleFocalPlane(num_detectors=8)
    fp_res = fp_task.run(from_images=bundle, groupby_keys=grouping_columns)
    for i in range(how_many):
        img = fp_res.data[i]
        if zscale:
            vmin, vmax = np.percentile(img.data.values, [5, 95])
        else:
            vmin, vmax = None, None
        img.show(show_det_id=True, with_mask=with_mask, vmin=vmin, vmax=vmax, **imshow_kwargs)
    return fp_res.data

In [ ]:
plot_focal_plane(mb_res.master_bias, cmap='gray')

### Preproc

Sanity checks for each step

In [ ]:
def quick_plot(img):
    vmin, vmax = np.percentile(img.data.values, [5, 95])
    img.show(cmap='gray', vmin=vmin, vmax=vmax, origin='lower')

In [ ]:
# plot master bias
quick_plot(mb_res.master_bias[4])

In [ ]:
## Loading a pair at an exptime
creator = ImageCreator(detector_config='../src/eregion/configs/detectors/deimos_sci.yaml', max_batch_size=2)

flpair = creator.run(input_source='/Users/yashvi/Desktop/Detector Characterization Tools/DTU_dettest/DTU_fullfp_bringup/PTC/SCI/20260718*/*SCI_*_flat_0.032*.fits',
                  identifier_func=guess_image_type_from_filename_DEIMOS,
                  fileloader_func=load_image_fits_DEIMOS,
                  data_on_demand=True)
flpair.data

In [ ]:
# quick_plot(flpair.data[0])
# plot_focal_plane(flpair.data, zscale=True, cmap='gray')

In [ ]:
# Do overscan sub
oscan_sub = ScanSubtraction(which_scan='serial_overscan', method='median_by_axis', trim_start=6)
flpair = oscan_sub.run(images=flpair.data)

In [ ]:
# quick_plot(after_osub.data[0])
# plot_focal_plane(flpair.data, zscale=True, cmap='gray')

In [ ]:
# Do sigma clip
cr_mask = SigmaClipMasking(sigma_clip_args={'sigma_lower':5.0, 'sigma_upper':5.0}) # set n_jobs=1 to be able to debug in pycharm inside parallelized jobs
flpair = cr_mask.run(images=flpair.data)

In [ ]:
# fpims = plot_focal_plane(flpair.data, zscale=True, with_mask=True, cmap='gray')

In [ ]:
# Do bias sub
bias_sub = BiasSubtraction(only_image_area=True)
flpair = bias_sub.run(images=flpair.data, master_bias=mb_res.master_bias)

In [ ]:
# quick_plot(after_bs.data[0])
# plot_focal_plane(flpair.data, zscale=True)

In [ ]:
# Do PTC
import eregion.tasks.ptc as ptc
import eregion
importlib.reload(eregion)
importlib.reload(ptc)

ptc_task = ptc.PTC()
pres = ptc_task.run(images=flpair.data)

In [ ]:
pres.ptc_table

In [ ]:
# save to fits
# pres.save(outpath)

In [ ]:
# load from fits
# pres = ptc.PTCResult.load(outpath)
# pres.ptc_table

For everything together, check out deimos_sci_testing.py

### From YAML pipeline config

In [ ]:
from eregion.pipeline import engine
import importlib
importlib.reload(engine)

eng = engine.PipelineEngine(pipeline_config_input='../src/eregion/configs/pipeline_flows/ptc_example.yaml',
                            runtime_variables={
                                'DETECTOR_CONFIG': '../src/eregion/configs/detectors/deimos_sci.yaml',
                                'BIAS_INPUT_SOURCE': '../data/deimos/sample_ptc/*bias*',
                                'INPUT_SOURCE': '../data/deimos/sample_ptc/*flat*',
                            })

In [ ]:
eng.run()

In [ ]:
eng.results['ptc_flow.make_ptc_table'].ptc_table